In [12]:
import os

os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import warnings
import logging


from dftorch import Constants, Structure, ESDriver


# Silence warnings and torch logs
warnings.filterwarnings("ignore")
os.environ["TORCH_LOGS"] = ""
os.environ["TORCHINDUCTOR_VERBOSE"] = "0"
os.environ["TORCHDYNAMO_VERBOSE"] = "0"
logging.getLogger("torch.fx").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.symbolic_shapes").setLevel(logging.CRITICAL)
logging.getLogger("torch.fx.experimental.recording").setLevel(logging.CRITICAL)

torch._dynamo.config.capture_dynamic_output_shape_ops = True
torch.set_default_dtype(torch.float64)
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


# Vibrational mode calculation.
### Done via finite differences. Displacements are batched.
### Solvation model introduces a significant overhead, so there is 'frozen_gbsa' option which uses frozen geometry but updated upon displacement charges.

In [13]:
%%time
dftorch_params = {
    "UNRESTRICTED": True,
    "SHARED_MU": False,
    "BROKEN_SYM": False,
    "coul_method": "!PME",
    "Coulomb_acc": 5e-5,
    "cutoff": 80.0,
    "PME_order": 4,
    "SCF_MAX_ITER": 100,
    "SCF_TOL": 1e-7,
    "SCF_ALPHA": 0.05,
    "KRYLOV_MAXRANK": 15,
    "KRYLOV_TOL": 1e-6,
    "KRYLOV_TOL_MD": 1e-4,
    "KRYLOV_START": 20,
    "d3_params": {"s6": 1.0, "s8": 0.5883, "a1": 0.5719, "a2": 3.6107},
    "solvent_param_file": "sk_orig/mio-1-1/mio-1-1/solvation/param_gbsa1_h2o.txt",
    "solvent": "water",
    "solvation_model": "gbsa",
    "h_damp_exp": 4.0,
    "ANDERSON_DEPTH": 15,
}

filename = "COORD_8WATER.xyz"
cell = None

const = Constants(
    filename,
    "sk_orig/3ob-3-1/",
    magnetic_hubbard_ldep=False,
    dftb3=True,
).to(device)

structure1 = Structure(
    filename,
    cell,
    const,
    charge=0,
    spin_pol=2,
    os=dftorch_params["UNRESTRICTED"],
    Te=1000.0,
    device=device,
    req_grad_xyz=False,
)

es_driver = ESDriver(
    dftorch_params, electronic_rcut=13.0, repulsive_rcut=6.0, device=device
)
es_driver(structure1, const, do_scf=True)
es_driver.calc_forces(structure1, const)

Ha = 0.0367493  # correct Ha → eV conversion
# print(f"  {'net spin':22s} {structure1.net_spin_sr.sum().item():12.4f}")
print(f"{'Energy Components':─^52}")
print(
    f"  {'E_total':22s} {structure1.e_tot * Ha:12.6f} Ha  ({structure1.e_tot:12.6f} eV)"
)
print(
    f"  {'E_band':22s} {structure1.e_band0 * Ha:12.6f} Ha ({structure1.e_band0:12.6f} eV)"
)
print(
    f"  {'E_coulomb':22s} {structure1.e_coul * Ha:12.6f} Ha ({structure1.e_coul:12.6f} eV)"
)
print(
    f"  {'E_repulsion':22s} {structure1.e_repulsion * Ha:12.6f} Ha ({structure1.e_repulsion:12.6f} eV)"
)
print(
    f"  {'E_entropy (-TS)':22s} {structure1.e_entropy * Ha:12.6f} Ha  ({structure1.e_entropy:12.6f} eV)"
)
print(
    f"  {'E_solvation':22s} {structure1.e_solv * Ha:12.6f} Ha  ({structure1.e_solv:12.6f} eV)"
)
print(f"  {'E_d3':22s} {structure1.e_d3 * Ha:12.6f} Ha  ({structure1.e_d3:12.6f} eV)")
print(
    f"  {'E_spin':22s} {structure1.e_spin.item() * Ha:12.6f} Ha  ({structure1.e_spin.item():12.6f} eV)"
)
print(f"{'─' * 52}")
print(
    f"  {'E_free (F=E-TS)':22s} {(structure1.e_tot - structure1.e_entropy) * Ha:12.6f} Ha ({(structure1.e_tot - structure1.e_entropy):12.6f} eV)"
)

DFTB3: True
coulomb_matrix_vectorized
  Coulomb_Real t 0.0 s
### Do _scf ###
  Initial dm_fermi

Starting cycle
Iter 1
Res = 1.305341530, dEc = 3.508114364, t = 0.0 s

Iter 2
Res = 0.932196317, dEc = 1.027848722, t = 0.0 s

Iter 3
Res = 1.317499922, dEc = 0.983231384, t = 0.0 s

Iter 4
Res = 1.173285554, dEc = 13.091805527, t = 0.0 s

Iter 5
Res = 2.053404756, dEc = 10.248587686, t = 0.0 s

Iter 6
Res = 1.462612464, dEc = 3.598874786, t = 0.0 s

Iter 7
Res = 1.611304562, dEc = 1.317651137, t = 0.0 s

Iter 8
Res = 1.526648719, dEc = 1.924947619, t = 0.0 s

Iter 9
Res = 1.384507796, dEc = 0.193160617, t = 0.0 s

Iter 10
Res = 1.367522368, dEc = 2.537628786, t = 0.0 s

Iter 11
Res = 1.156579122, dEc = 1.460655704, t = 0.0 s

Iter 12
Res = 1.328961505, dEc = 0.059254525, t = 0.0 s

Iter 13
Res = 1.322808893, dEc = 2.032563534, t = 0.0 s

Iter 14
Res = 0.926183823, dEc = 0.028175297, t = 0.0 s

Iter 15
Res = 0.889299541, dEc = 0.000992442, t = 0.0 s

Iter 16
Res = 0.889665052, dEc = 0.10157

In [ ]:
es_driver.calc_hessian(
    structure1,
    const,
    mode="full",  # "full" | "frozen_gbsa"
    batch_size=64,
    delta=1e-6,
    filename=filename,
    verbose=True,
);

FD Hessian (full): 24 atoms, 72 DOFs, δ = 1e-06 Å
  batch_size=64 → 32 DOFs/batch, 3 batched ES calls
GBSA initialization time: 2.09 seconds
  Using q_init (skipping initial dm_fermi)

Starting cycle
Iter 1
Batch 0: Res = 8.555e-01, dEc = 2.500e+00
Batch 1: Res = 8.555e-01, dEc = 2.500e+00
Batch 2: Res = 8.555e-01, dEc = 2.500e+00
Batch 3: Res = 8.555e-01, dEc = 2.500e+00
Batch 4: Res = 8.555e-01, dEc = 2.500e+00
Batch 5: Res = 8.555e-01, dEc = 2.500e+00
Batch 6: Res = 8.555e-01, dEc = 2.500e+00
Batch 7: Res = 8.555e-01, dEc = 2.500e+00
Batch 8: Res = 8.555e-01, dEc = 2.500e+00
Batch 9: Res = 8.555e-01, dEc = 2.500e+00
Batch 10: Res = 8.555e-01, dEc = 2.500e+00
Batch 11: Res = 8.555e-01, dEc = 2.500e+00
Batch 12: Res = 8.555e-01, dEc = 2.500e+00
Batch 13: Res = 8.555e-01, dEc = 2.500e+00
Batch 14: Res = 8.555e-01, dEc = 2.500e+00
Batch 15: Res = 8.555e-01, dEc = 2.500e+00
Batch 16: Res = 8.555e-01, dEc = 2.500e+00
Batch 17: Res = 8.555e-01, dEc = 2.500e+00
Batch 18: Res = 8.555e-01, dE

In [16]:
freqs_cm, eigvecs, results = ESDriver.calc_frequencies(structure1, proj_tr=True)

N_at = structure1.Nats
n3 = 3 * N_at

print(f"{'Vibrational Frequencies':─^60}")
print(f"  N_atoms = {N_at},  3N = {n3},  6 T+R modes expected\n")
print(f"  {'Mode':>5s}  {'Eigenvalue':>14s}  {'Freq (cm⁻¹)':>14s}  {'Note':>8s}")
print(f"  {'─' * 50}")
for k in range(n3):
    note = ""
    if abs(freqs_cm[k]) < 10:
        note = "T/R"
    elif freqs_cm[k] < -10:
        note = "imag"
    print(
        f"  {k + 1:5d}  {results['eigenvalues'][k]:14.6f}  {freqs_cm[k]:14.2f}  {note:>8s}"
    )

print(
    f"\n  Conversion factor: {results['conversion_factor']:.3f} cm⁻¹ / sqrt(eV/(Å²·amu))"
)
print(f"  6 lowest freqs: {freqs_cm[:6]}")
print(f"  Highest freq:   {freqs_cm[-1]:.2f} cm⁻¹")
print(f"  Imaginary modes (< -10 cm⁻¹): {results['n_imag']}")

──────────────────Vibrational Frequencies───────────────────
  N_atoms = 24,  3N = 72,  6 T+R modes expected

   Mode      Eigenvalue     Freq (cm⁻¹)      Note
  ──────────────────────────────────────────────────
      1       -9.495519        -1606.90      imag
      2       -8.038197        -1478.46      imag
      3       -6.321988        -1311.16      imag
      4       -4.773397        -1139.32      imag
      5       -3.401297         -961.73      imag
      6       -1.473435         -632.99      imag
      7       -1.063994         -537.90      imag
      8       -0.399515         -329.61      imag
      9       -0.236578         -253.64      imag
     10       -0.214971         -241.78      imag
     11       -0.020499          -74.66      imag
     12       -0.013162          -59.83      imag
     13       -0.011885          -56.85      imag
     14       -0.008038          -46.75      imag
     15       -0.007380          -44.80      imag
     16       -0.006234          -41.